# DYRK1B pipeline v2 (Colab)
Changes vs v1: non-azaindoles ranked by a model trained **without** azaindoles; cluster-split CV; docking pool rebuilt; final list = ML 0.5 + docking 0.4 + hinge 0.1 with **max 2 picks per cluster** and max 10 azaindoles.
Run cells in order. Output: `submission_v2.txt`. Docking results from v1 (`dock_results.csv`) are reused if the file is still there.

In [ ]:
# ===== 1. INSTALL + DATA =====
!pip -q install rdkit scikit-learn pandas numpy scipy
!apt-get -qq install -y openbabel > /dev/null
!wget -q -nc https://files.rcsb.org/download/8C2Z.pdb
!wget -q -nc -O vina https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64
!chmod +x vina
!git clone -q https://github.com/avnikonenko/Drug-Design-workshop-challenge-2026 2>/dev/null

import os
print("vina:", os.popen("./vina --version 2>&1").read().strip())
print("8C2Z:", os.path.exists("8C2Z.pdb"), "| obabel:", os.system("which obabel > /dev/null") == 0)

In [ ]:
# ===== V2-A: ML v2 (azaindole down-weighting + cluster-split CV) =====
import numpy as np, pandas as pd
from scipy.stats import rankdata
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator as rfg
from rdkit.ML.Cluster import Butina
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
RDLogger.DisableLog("rdApp.*")

D = "Drug-Design-workshop-challenge-2026/data/"
tr = pd.read_csv(D + "chembl_stereo_classes_fixed.smi", sep="\t", header=None, names=["smiles", "id", "act"])
bl = pd.read_csv(D + "blind_stereo.smi", sep="\t", header=None, names=["smiles", "id"])
tr["y"] = tr.act.astype(str).str.lower().isin(["1", "1.0", "active"]).astype(int)

morgan  = rfg.GetMorganGenerator(radius=2, fpSize=2048)
fmorgan = rfg.GetMorganGenerator(radius=2, fpSize=2048, atomInvariantsGenerator=rfg.GetMorganFeatureAtomInvGen())
apair   = rfg.GetAtomPairGenerator(fpSize=2048)
AZ = Chem.MolFromSmarts("c1cnc2[nH]ccc2c1")

def feats(smiles):
    bits, X, az, keep = [], [], [], []
    for i, s in enumerate(smiles):
        m = Chem.MolFromSmiles(s)
        if m is None: continue
        bits.append(morgan.GetFingerprint(m)); keep.append(i); az.append(m.HasSubstructMatch(AZ))
        X.append(np.concatenate([morgan.GetCountFingerprintAsNumPy(m),
                                 fmorgan.GetCountFingerprintAsNumPy(m),
                                 apair.GetCountFingerprintAsNumPy(m)]).astype(np.float32))
    return bits, np.vstack(X), np.array(az), keep

tb, Xt, taz, kt = feats(tr.smiles); y = tr.y.values[kt]
bb, Xb, baz, kb = feats(bl.smiles)
bl = bl.iloc[kb].reset_index(drop=True)
print(f"train {len(y)} (actives {y.sum()}, azaindoles {taz.sum()}, azaindole actives {(taz & (y==1)).sum()})")

# cluster labels for honest CV (compounds in the same cluster never split across folds)
d = []
for i in range(1, len(tb)):
    d += [1 - s for s in DataStructs.BulkTanimotoSimilarity(tb[i], tb[:i])]
groups = np.zeros(len(tb), int)
for ci, c in enumerate(Butina.ClusterData(d, len(tb), 0.6, isDistData=True)):
    groups[list(c)] = ci

AZ_W = 0.3                                   # azaindole training weight
w = np.where(taz, AZ_W, 1.0)

def fit(X, y, w, seed=0):
    rf = RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample", n_jobs=-1, random_state=seed)
    et = ExtraTreesClassifier(n_estimators=500, class_weight="balanced_subsample", n_jobs=-1, random_state=seed)
    return [rf.fit(X, y, sample_weight=w), et.fit(X, y, sample_weight=w)]

def pred(models, X): return np.mean([m.predict_proba(X)[:, 1] for m in models], axis=0)

for name, ww in [("equal weights", np.ones(len(y))), (f"azaindole weight {AZ_W}", w)]:
    aucs, aucs_nonaz = [], []
    for a, b in GroupKFold(4).split(Xt, y, groups):
        p = pred(fit(Xt[a], y[a], ww[a]), Xt[b])
        if len(set(y[b])) > 1: aucs.append(roc_auc_score(y[b], p))
        m_ = ~taz[b]
        if len(set(y[b][m_])) > 1: aucs_nonaz.append(roc_auc_score(y[b][m_], p[m_]))
    print(f"cluster-CV | {name:22s} AUC all {np.mean(aucs):.3f} | AUC non-azaindoles {np.mean(aucs_nonaz):.3f}")

# --- final scoring: non-azaindoles ranked by a model trained WITHOUT azaindoles ---
na = ~taz
models_na = fit(Xt[na], y[na], np.ones(na.sum()))
act_na   = [b for b, l, z in zip(tb, y, taz) if l == 1 and not z]
inact_na = [b for b, l, z in zip(tb, y, taz) if l == 0 and not z]
def sim_na(fp, k=3):
    a = np.sort(DataStructs.BulkTanimotoSimilarity(fp, act_na))[::-1][:k].mean()
    i = np.sort(DataStructs.BulkTanimotoSimilarity(fp, inact_na))[::-1][:k].mean()
    return a - 0.3 * i
ml_na = pred(models_na, Xb)
sm_na = np.array([sim_na(f) for f in bb])
s_na = (rankdata(ml_na) + rankdata(sm_na)) / (2 * len(bl))

# azaindoles keep the full model (they are capped later anyway)
models_all = fit(Xt, y, w)
ml_all = pred(models_all, Xb)
act = [b for b, l in zip(tb, y) if l == 1]; inact = [b for b, l in zip(tb, y) if l == 0]
def sim_all(fp, k=3):
    a = np.sort(DataStructs.BulkTanimotoSimilarity(fp, act))[::-1][:k].mean()
    i = np.sort(DataStructs.BulkTanimotoSimilarity(fp, inact))[::-1][:k].mean()
    return a - 0.3 * i
s_all = (rankdata(ml_all) + rankdata(np.array([sim_all(f) for f in bb]))) / (2 * len(bl))

score = pd.Series(np.nan, index=range(len(bl)))
idx_n, idx_a = np.where(~baz)[0], np.where(baz)[0]
score.iloc[idx_n] = rankdata(s_na[idx_n]) / len(idx_n)               # 0..1 among non-azaindoles
score.iloc[idx_a] = 0.5 + 0.5 * rankdata(s_all[idx_a]) / len(idx_a)  # 0.5..1 among azaindoles
bl["score"] = score.values
bl["az"] = baz
bl = bl.sort_values("score", ascending=False).reset_index(drop=True)
bl.drop(columns="az").to_csv("ranked_blind_set_v2.csv", index=False)
print("saved ranked_blind_set_v2.csv | blind non-azaindoles:", len(idx_n), "| azaindoles:", len(idx_a))
print("azaindoles in top 100:", int(bl.head(100).az.sum()), "| top 300:", int(bl.head(300).az.sum()))

In [ ]:
# ===== 3. RECEPTOR, DOCKING BOX, HINGE ATOMS, DOCKING POOL =====
import numpy as np, pandas as pd, subprocess
from collections import defaultdict
from rdkit import Chem

lines = [l for l in open("8C2Z.pdb") if l.startswith(("ATOM", "HETATM"))]
xyz = lambda l: np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])

prot = [l for l in lines if l.startswith("ATOM") and l[21] == "A" and l[16] in " A"]
open("rec_clean.pdb", "w").write("".join(prot) + "END\n")
subprocess.run("obabel rec_clean.pdb -O rec.pdbqt -xr -p 7.4", shell=True, check=True)

skip = {"HOH","DMS","GOL","EDO","SO4","PO4","CL","NA","MG","ZN","K","CA","PEG","ACT"}
groups = defaultdict(list)
for l in lines:
    if l.startswith("HETATM") and l[17:20].strip() not in skip:
        groups[(l[17:20], l[21], l[22:26])].append(l)
key = max(groups, key=lambda k: len(groups[k]))
lig = groups[key]
lc = np.array([xyz(l) for l in lig])
center = lc.mean(0)
size = np.minimum(np.maximum(lc.max(0) - lc.min(0) + 12, 20), 22)   # smaller box = faster

lig_no = np.array([xyz(l) for l in lig if l[76:78].strip() in ("N", "O")])
hinge = np.array([xyz(l) for l in prot if l[12:16].strip() in ("N", "O")
                  and np.min(np.linalg.norm(lig_no - xyz(l), axis=1)) < 3.4])
print("reference ligand:", key, len(lig), "atoms | center", center.round(1), "box", size.round(0))
print("hinge atoms found:", len(hinge), "(need >= 2)")

# docking pool (v2): top 500 non-azaindoles + best 20 azaindoles from the v2 ranking
r = pd.read_csv("ranked_blind_set_v2.csv")
azq = Chem.MolFromSmarts("c1cnc2[nH]ccc2c1")
r["az"] = [Chem.MolFromSmiles(s).HasSubstructMatch(azq) for s in r.smiles]
pool = pd.concat([r[~r.az].head(500), r[r.az].head(20)]).sort_values("score", ascending=False)
pool[["smiles", "id", "score"]].to_csv("dock_pool.csv", index=False)
print("pool:", len(pool), "| azaindoles:", int(pool.az.sum()))

In [ ]:
# ===== V2-B2. DOCKING (reuses dock_results.csv; skips docked ids) (stops itself at TIME_LIMIT; re-run to continue) =====
import os, time, subprocess, numpy as np, pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, rdFingerprintGenerator as rfg
from rdkit.ML.Cluster import Butina

N_DOCK     = 150        # how many compounds to dock in total
EXHAUST    = 1          # 1 = fast, 4 = more accurate but ~4x slower
TIME_LIMIT = 15 * 60    # seconds per run of this cell

pool = pd.read_csv("dock_pool.csv").reset_index(drop=True)
gen = rfg.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in pool.smiles]
d = []
for i in range(1, len(fps)):
    d += [1 - x for x in DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])]
cl = Butina.ClusterData(d, len(fps), 0.65, isDistData=True)
reps = sorted(min(c) for c in cl)                       # best member of each cluster first
order = reps + [i for i in range(len(pool)) if i not in set(reps)]
todo = pool.iloc[order[:N_DOCK]]

def prep(smi):
    m = Chem.AddHs(Chem.MolFromSmiles(smi))
    if AllChem.EmbedMolecule(m, randomSeed=7) != 0: return False
    try: AllChem.MMFFOptimizeMolecule(m, maxIters=200)
    except Exception: pass
    Chem.MolToMolFile(m, "lig.mol")
    if os.path.exists("lig.pdbqt"): os.remove("lig.pdbqt")
    subprocess.run("obabel lig.mol -O lig.pdbqt", shell=True, capture_output=True)
    return os.path.exists("lig.pdbqt") and os.path.getsize("lig.pdbqt") > 0

def dock():
    if os.path.exists("out.pdbqt"): os.remove("out.pdbqt")
    cmd = ["./vina", "--receptor", "rec.pdbqt", "--ligand", "lig.pdbqt",
           "--center_x", str(center[0]), "--center_y", str(center[1]), "--center_z", str(center[2]),
           "--size_x", str(size[0]), "--size_y", str(size[1]), "--size_z", str(size[2]),
           "--exhaustiveness", str(EXHAUST), "--num_modes", "3", "--cpu", "2",
           "--seed", "7", "--out", "out.pdbqt"]
    subprocess.run(cmd, capture_output=True, timeout=600)
    if not os.path.exists("out.pdbqt"): return None
    txt = open("out.pdbqt").read()
    res = [l for l in txt.splitlines() if l.startswith("REMARK VINA RESULT")]
    return (float(res[0].split()[3]), txt) if res else None

def hinge_hit(poses):
    for model in poses.split("ENDMDL"):
        pts = [xyz(l) for l in model.splitlines()
               if l.startswith(("ATOM", "HETATM")) and l.split()[-1] in ("N","NA","NS","OA","OS")]
        if pts and len(hinge) and np.linalg.norm(hinge[:, None] - np.array(pts)[None], axis=2).min() < 3.5:
            return 1
    return 0

done = pd.read_csv("dock_results.csv") if os.path.exists("dock_results.csv") else pd.DataFrame(columns=["id","aff","hinge"])
rows, seen = done.to_dict("records"), set(done.id)
print("already docked:", len(seen))

t0 = time.time()
for k, r_ in enumerate(todo.itertuples()):
    if r_.id in seen: continue
    if time.time() - t0 > TIME_LIMIT:
        print("time limit reached - re-run this cell to continue"); break
    try:
        if not prep(r_.smiles): continue
        out = dock()
        if out is None: continue
        rows.append({"id": r_.id, "aff": out[0], "hinge": hinge_hit(out[1])})
    except Exception as e:
        print(r_.id, "failed:", str(e)[:60])
    if k % 5 == 0:
        pd.DataFrame(rows).to_csv("dock_results.csv", index=False)
        print(f"{k+1}/{len(todo)}  {time.time()-t0:.0f}s  docked so far: {len(rows)}")
pd.DataFrame(rows).to_csv("dock_results.csv", index=False)
print("docked total:", len(rows))

In [ ]:
# ===== V2-C: BLEND + CLUSTER CAP -> submission_v2.txt =====
import numpy as np, pandas as pd, os
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator as rfg
from rdkit.ML.Cluster import Butina

W_ML, W_DOCK, W_HINGE = 0.5, 0.4, 0.1
MAX_AZ, MAX_PER_CLUSTER, CLUSTER_DIST = 10, 2, 0.6

AZ = Chem.MolFromSmarts("c1cnc2[nH]ccc2c1")
gen = rfg.GetMorganGenerator(radius=2, fpSize=2048)
ml = pd.read_csv("ranked_blind_set_v2.csv")                       # smiles, id, score
dk = pd.read_csv("dock_results.csv") if os.path.exists("dock_results.csv") else pd.DataFrame(columns=["id","aff","hinge"])
pool = pd.read_csv("dock_pool.csv")[["id"]]
cand = ml.merge(pool, on="id")                                    # only compounds in the docking pool
cand = cand.merge(dk, on="id", how="left")
has = cand.aff.notna()
print("pool:", len(cand), "| docked:", int(has.sum()))

# docked compounds: blend; undocked: ML only (scaled to the same 0-1 range, small penalty for no docking evidence)
cand["ml_r"] = cand.score.rank(pct=True)
cand["dock_r"] = np.nan
cand.loc[has, "dock_r"] = (-cand.loc[has, "aff"]).rank(pct=True)
cand["hinge"] = cand.hinge.fillna(0)
cand["final"] = np.where(has, W_ML * cand.ml_r + W_DOCK * cand.dock_r + W_HINGE * cand.hinge,
                              (W_ML + W_DOCK * 0.5) * cand.ml_r)   # undocked: assume average docking rank
cand["az"] = [Chem.MolFromSmiles(s).HasSubstructMatch(AZ) for s in cand.smiles]
cand["fp"] = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in cand.smiles]
cand = cand.sort_values("final", ascending=False).reset_index(drop=True)

# cluster the candidates; allow at most MAX_PER_CLUSTER picks per cluster
d = []
for i in range(1, len(cand)):
    d += [1 - x for x in DataStructs.BulkTanimotoSimilarity(cand.fp[i], list(cand.fp[:i]))]
cl_of = np.zeros(len(cand), int)
for ci, c in enumerate(Butina.ClusterData(d, len(cand), CLUSTER_DIST, isDistData=True)):
    cl_of[list(c)] = ci
cand["cluster"] = cl_of

ids, n_az, per = [], 0, {}
for r in cand.itertuples():
    if r.az and n_az >= MAX_AZ: continue
    if per.get(r.cluster, 0) >= MAX_PER_CLUSTER: continue
    ids.append(r.id); n_az += int(r.az); per[r.cluster] = per.get(r.cluster, 0) + 1
    if len(ids) == 100: break
if len(ids) < 100:                                                # relax the cluster cap if needed
    for r in cand.itertuples():
        if r.id in ids: continue
        if r.az and n_az >= MAX_AZ: continue
        ids.append(r.id); n_az += int(r.az)
        if len(ids) == 100: break

open("submission_v2.txt", "w").write("\n".join(ids) + "\n")
sel = cand[cand.id.isin(ids)]
print("IDs:", len(ids), "unique:", len(set(ids)), "| azaindoles:", n_az,
      "| clusters used:", sel.cluster.nunique(), "| docked among picks:", int(sel.aff.notna().sum()))

In [ ]:
# ===== CHECK =====
from google.colab import files
ids = open("submission_v2.txt").read().split()
print(len(ids), len(set(ids)), ids[:3])
files.download("submission_v2.txt")